In [1]:
import os
import sys

from dotenv import load_dotenv

load_dotenv()

print(f"실행 경로: {sys.executable}")
if ".venv" in sys.executable:
    print("[OK] .venv 가상환경 사용 중")
else:
    print("[WARN] .venv 가상환경이 아닌 인터프리터가 선택되어 있습니다.")
    
print("[OK] OpenAI API 키 설정됨" if os.getenv("OPENAI_API_KEY") else "[ERROR] OpenAI API 키 미설정")
print("[OK] LangSmith 키 설정됨" if os.getenv("LANGSMITH_API_KEY") else "[ERROR] LangSmith 키 미설정")

실행 경로: d:\yugyeong\rag\.venv\Scripts\python.exe
[OK] .venv 가상환경 사용 중
[OK] OpenAI API 키 설정됨
[OK] LangSmith 키 설정됨


In [91]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

In [51]:
PDF_PATH = "./data/ex_ai_guide.pdf"

loader = PyPDFLoader(PDF_PATH, extract_images=True)

docs = loader.load()

[INFO] 2026-09-30 10:43:29,814 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-30 10:43:29,824 [RapidOCR] download_file.py:60: File exists and is valid: D:\yugyeong\rag\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-30 10:43:29,826 [RapidOCR] main.py:63: Using D:\yugyeong\rag\.venv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-09-30 10:43:29,993 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-30 10:43:29,996 [RapidOCR] download_file.py:60: File exists and is valid: D:\yugyeong\rag\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-30 10:43:29,997 [RapidOCR] main.py:63: Using D:\yugyeong\rag\.venv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-09-30 10:43:30,140 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-09-30 10:43:30,198 [RapidOCR] download_file.py:60: File exists and is valid: D:\yugyeong\

In [110]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

split_documents = text_splitter.split_documents(docs)

In [79]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [80]:
vectorstore = Chroma.from_documents(
    split_documents,
    embeddings,
    persist_directory = './db/chromadb',
    collection_metadata = {'hnsw:space': 'cosine'}
)

In [118]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [119]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
prompt = PromptTemplate.from_template(
    """당신은 정확한 답변을 제공하는 전문가입니다.
다음 규칙을 반드시 따르세요:
1. 아래 제공된 문서만을 근거로 답변하세요.
2. 문서에 관련 정보가 없으면 "해당 정보가 제공된 문서에 없습니다"라고 답변하세요.
3. 추측이나 외부 지식을 사용하지 마세요.

#Context: 
{context}

#Question:
{question}

"""
)

PydanticUserError: `FlashrankRerank` is not fully defined; you should define `Ranker`, then call `FlashrankRerank.model_rebuild()`.

For further information visit https://errors.pydantic.dev/2.13/u/class-not-fully-defined

In [ ]:
chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | ChatOpenAI(model="gpt-5-mini", temperature=0)
    | StrOutputParser()
)

NameError: name 'compression_retriever' is not defined

In [124]:
question = "생성형 AI가 만든 창작물의 저작권 문제와 관련하여 왜 법적 기준이 명확하지 않은지 설명하세요."
response = chain.invoke(question)
print(response)

법적 기준이 명확하지 않은 이유는 여러 가지가 복합적으로 작용하기 때문입니다.

1) 저작권 제도 자체가 인간 창작자를 전제로 함  
   대부분 국가의 저작권법은 ‘사람(자연인)의 창작적 표현’에 권리를 부여하도록 설계되어 있습니다. 생성형 AI가 스스로 “저자”인지, 혹은 저작권을 가질 수 있는지는 기존 법 체계로는 설명하기 어렵습니다.

2) 학습 데이터의 출처와 구성 불투명성  
   생성형 AI는 웹·문서·이미지 등 방대한 데이터로 학습하는데, 그 데이터에 저작권 있는 자료가 포함되어 있는지, 어느 부분이 학습에 기여했는지를 추적하기 어렵습니다. 따라서 결과물이 특정 저작물을 실질적으로 복제·파생한 것인지 증명하기 힘듭니다.

3) 산출물의 성격(모방·변형·혼합)의 경계가 불명확  
   모델 출력은 여러 자료를 결합·일반화한 ‘합성적’ 결과인 경우가 많아, 어디까지가 원저작물의 표현을 그대로 따른 복제이고 어디부터가 새로운 표현인지 판단하기 어렵습니다.

4) 저작권 귀속 주체의 불명확성  
   저작권을 누가 가지는지(모델 개발자, 모델을 운영·제공한 사업자, 프롬프트를 입력한 사용자 등)도 법적 쟁점입니다. 각 주체별 책임·권리 범위를 규정하는 판례나 법 규정이 부족합니다.

5) 관할권·법리 차이와 판례 부재  
   공정사용(fair use) 등 예외규정과 해석이 국가마다 다르고, 생성형 AI 관련 판례가 아직 충분치 않아 일관된 법리 정립이 이루어지지 않았습니다.

6) 기술적 특성(암묵적 일반화 vs. 기억화)의 증명 어려움  
   모델이 단순히 원문을 암기·출력한 것인지, 학습을 통해 일반화한 결과인지 기술적으로 증명하기 어려워 침해 판단이 복잡합니다.

결과적으로 법·규범·기술·증거(증명) 측면이 모두 불명확하게 얽혀 있어 현재로서는 일관된 법적 기준을 확립하기 어렵고, 이에 따라 소송과 논쟁이 이어지고 있습니다.
